# 04 · VLM scorer and verifier (M6, E7)
Use a **fresh A100 runtime**: vLLM installs its own PyTorch build. Budget: a full VLM-scorer run over all test
executions should take < 2 h; runtime per video and JSON parse failures are written to each run's `metrics.json`.

## Setup
Mounts Drive, clones or updates the repo, installs it and sets the `SOPVERIFY_*` paths. Safe to re-run after a runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%bash
set -e
REPO=/content/SOP-Verification
BRANCH=claude/nifty-mccarthy-5suut3   # switch to main once the branch is merged
if [ -d $REPO/.git ]; then
  git -C $REPO fetch -q origin $BRANCH && git -C $REPO checkout -q $BRANCH && git -C $REPO pull -q --ff-only origin $BRANCH
else
  git clone -q -b $BRANCH https://github.com/RohanSada/SOP-Verification.git $REPO
fi
pip install -q -e "$REPO[app,vlm,dev]"
git -C $REPO log --oneline -1

In [ ]:
import os, sys
sys.path.insert(0, "/content/SOP-Verification/src")   # editable install without a runtime restart
os.chdir("/content/SOP-Verification")
from sopverify.colab import setup_env, show
ROOT = "/content/drive/MyDrive/sopverify"
ENV = setup_env(ROOT)
try:  # optional Hugging Face token (Colab secret named HF_TOKEN), in case the dataset is gated
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
ENV

### Install vLLM
Takes a few minutes and replaces PyTorch. When it finishes: **Runtime → Restart session**, then run the Setup cells again (skip this install cell).

In [ ]:
!pip install -q vllm
!pip show vllm torch | grep -E '^(Name|Version)'

### Verify the model id

In [ ]:
from huggingface_hub import model_info
from sopverify.config import load_config
cfg = load_config()
info = model_info(cfg.vlm.model_id); print(info.id, info.sha, info.pipeline_tag)

### Smoke test: one keyframe

In [ ]:
import json
from sopverify.data.impact.catalog import Catalog
from sopverify.procedure.loader import load_procedure
from sopverify.vlm.backend import make_backend
from sopverify.vlm.frames import VideoFrameSource
from sopverify.vlm.prompts import scorer_request
cat = Catalog.from_paths(os.environ["SOPVERIFY_DATA_ROOT"], None)
proc = load_procedure("procedures/impact_reassembly_A")
backend = make_backend(cfg.vlm)
frames = VideoFrameSource(proc.roi.box, proc.roi.frame_size, cfg.vlm.max_long_side)
ref = cat.ref(cat.execution_ids()[0])
img = frames.crops(ref.video_path, [ref.duration_s - 1], ref.fps)[0]
print(backend.generate([scorer_request(proc.steps("grouped"), img, ref.duration_s - 1)])[0])

### E7

In [ ]:
import time
from sopverify.eval.runner import run_experiment
t0 = time.time()
res = run_experiment("configs/experiments/E7_vlm.yaml", vlm_backend=backend)
print(f"total {time.time() - t0:.0f}s")
for v, m in res.items():
    print(v, m["runtime_s"], m.get("vlm"))

In [ ]:
!sopverify tables --experiments E7_vlm --out "$SOPVERIFY_EXPORTS/RESULTS_E7.md"
print(open(os.environ["SOPVERIFY_EXPORTS"] + "/RESULTS_E7.md").read())